# LLM Reasoning Drift Analysis Claude Debugging Session

Analysis of a Claude Code conversation transcript to identify where and how
AI reasoning shifted from evidence-grounded claims to unsupported assertions
during a complex software debugging task (JavaScript AMD module loading bug
in a Ruby on Rails / Hyrax application).

**Source file:** `llm-reasoning-colab-analysis-20260715.jsonl`
**Session ID:** `828a165e-2bd0-4e8d-9564-67ac87d6dd54`
**Lines in transcript:** 11,177 (everything is raw debugging)

In [ ]:
WORKING_MODEL = 'gemini-3.1-flash-lite'

def annotate_gemini(prior_tool, prior_msg, current_msg, system_prompt):
    response = gemini_client.models.generate_content(
        model=WORKING_MODEL,
        contents=build_user_prompt(prior_tool, prior_msg, current_msg),
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0,
            response_mime_type='application/json'
        )
    )
    return json.loads(response.text)

In [ ]:
# Cell 2 Packages not pre-installed in Colab
!pip install -q sentence-transformers hmmlearn google-genai

In [ ]:
#Cell 3 Imports

import json
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.metrics import (
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
from hmmlearn import hmm

from google import genai
from google.genai import types

from google.colab import drive, userdata
from IPython.display import display

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_rows', 100)

print("All imports successful.")

In [ ]:
#Cell 4 Mount Google Drive

drive.mount('/content/drive', force_remount=True)

GEMINI_API_KEY = userdata.get('llm-reasoning-colab-analysis-gemini')
gemini_client  = genai.Client(api_key=GEMINI_API_KEY)
WORKING_MODEL = 'gemini-3.1-flash-lite'

for attempt in range(5):
    try:
        test = gemini_client.models.generate_content(
            model=WORKING_MODEL,
            contents='Reply with the single word: connected'
        )
        print(f"Gemini ({WORKING_MODEL}): {test.text.strip()}")
        break
    except Exception as e:
        if '503' in str(e) and attempt < 4:
            print(f"Overloaded, retrying in 10s... (attempt {attempt + 1}/5)")
            time.sleep(10)
        else:
            raise

TRANSCRIPT_PATH = '/content/drive/MyDrive/session-828a165e-full-transcript-20260715.jsonl'
print("Transcript path set.")

In [ ]:
#Cell 5 Verify file before parsing

import os

TRANSCRIPT_PATH = '/content/drive/MyDrive/session-828a165e-full-transcript-20260715.jsonl'

size_mb = os.path.getsize(TRANSCRIPT_PATH) / (1024 * 1024)
with open(TRANSCRIPT_PATH, 'r') as f:
    line_count = sum(1 for _ in f)

print(f"File size:  {size_mb:.1f} MB")
print(f"Line count: {line_count:,}")

with open(TRANSCRIPT_PATH, 'r') as f:
    first_record = json.loads(f.readline())

print("\nTop-level keys in first record:")
print(list(first_record.keys()))

print("\nFull first record (formatted):")
print(json.dumps(first_record, indent=2)[:1000])

In [ ]:
#Cell 6 Annotation functions (Gemini + Anthropic)

LABELS_ORDER = [
    'GROUNDED', 'INFERRED', 'ASSERTED',
    'CORRECTING', 'ACTION', 'MIXED'
]

def build_user_prompt(prior_tool, prior_msg, current_msg):
    return f"""
---PRIOR TOOL RESULT (if any)---
{prior_tool or 'none'}

---PRIOR ASSISTANT MESSAGE---
{prior_msg or 'none'}

---CURRENT MESSAGE TO CLASSIFY---
{current_msg}

Respond with JSON only.
"""

def annotate_gemini(prior_tool, prior_msg, current_msg, system_prompt):
    response = gemini_client.models.generate_content(
        model=WORKING_MODEL,
        contents=build_user_prompt(prior_tool, prior_msg, current_msg),
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0,
            response_mime_type='application/json'
        )
    )
    return json.loads(response.text)

def annotate_with_retry(prior_tool, prior_msg, current_msg,
                        system_prompt, max_retries=3, delay=5):
    for attempt in range(max_retries):
        try:
            return annotate_gemini(
                prior_tool, prior_msg, current_msg, system_prompt
            )
        except Exception as e:
            if '503' in str(e) and attempt < max_retries - 1:
                print(f"  Overloaded, retrying in {delay}s...")
                time.sleep(delay)
            else:
                raise

print("Annotation functions ready.")

In [ ]:
#Cell 7 Determinism check

def check_determinism(sample_df, system_prompt, n_runs=3, sample_size=20):
    test_sample = sample_df.sample(
        min(sample_size, len(sample_df)), random_state=42
    )
    results = []

    for i, (_, row) in enumerate(test_sample.iterrows()):
        print(f"  Testing message {i+1}/{len(test_sample)}...", end='\r')
        labels = []
        for _ in range(n_runs):
            ann = annotate_with_retry(
                row['prior_tool_result'],
                row['prior_assistant_content'],
                row['content'],
                system_prompt
            )
            labels.append(ann['classification'])
            time.sleep(1)

        all_same = len(set(labels)) == 1
        results.append({
            'message_id':    row['message_id'],
            'labels':        labels,
            'deterministic': all_same,
            'content':       row['content'][:150]
        })

    det_df   = pd.DataFrame(results)
    det_rate = det_df['deterministic'].mean()

    print(f"\nDeterminism rate: {det_rate:.1%}")
    print(f"Consistent:       {det_df['deterministic'].sum()}/{len(det_df)}")

    if not det_df[~det_df['deterministic']].empty:
        print("\nAmbiguous messages — these sit on a category boundary:")
        display(det_df[~det_df['deterministic']][
            ['message_id', 'labels', 'content']
        ])

    return det_rate, det_df

print("Determinism check ready.")

In [ ]:
#Cell 8 cross-model Kappa (Anthropic vs Gemini)

def check_logic_rules(annotation):
    violations = []
    c  = annotation['classification']
    ep = annotation['evidence_present']
    es = annotation['evidence_source']
    cf = annotation['confidence']
    r  = annotation.get('reasoning', '')

    if c == 'GROUNDED' and not ep:
        violations.append('GROUNDED_NO_EVIDENCE')
    if c == 'ASSERTED' and es == 'tool result':
        violations.append('ASSERTED_CLAIMS_TOOL_EVIDENCE')
    if c == 'CORRECTING' and not any(
        w in r.lower() for w in
        ['wrong','incorrect','revert','mistaken','error','misidentified']
    ):
        violations.append('CORRECTING_NO_REVISION_LANGUAGE')
    if c == 'GROUNDED' and cf < 0.7:
        violations.append('GROUNDED_LOW_CONFIDENCE')
    if es == 'session summary' and c == 'GROUNDED':
        violations.append('SESSION_SUMMARY_LABELED_GROUNDED')

    return violations

def run_logic_checks(ann_df):
    ann_df = ann_df.copy()
    ann_df['violations']    = ann_df['annotation'].apply(check_logic_rules)
    ann_df['has_violation'] = ann_df['violations'].apply(lambda x: len(x) > 0)

    violation_rate = ann_df['has_violation'].mean()
    print(f"Violation rate: {violation_rate:.1%}")

    all_v = [v for vlist in ann_df['violations'] for v in vlist]
    print("\nBreakdown:")
    for v, count in Counter(all_v).most_common():
        print(f"  {v}: {count}")

    return violation_rate, ann_df

print("Logic checks ready.")

In [ ]:
#Cell 9 Logic rule validation

def all_checks_pass(scores):
    return (
        scores['determinism']     > 0.95 and
        scores['violation_rate']  < 0.05 and
        scores['asserted_recall'] > 0.65 and
        scores['low_conf_rate']   < 0.20
    )

def print_validation_summary(scores):
    checks = [
        ('Determinism',     scores['determinism'],     0.95, 'above'),
        ('Violation rate',  scores['violation_rate'],  0.05, 'below'),
        ('ASSERTED recall', scores['asserted_recall'], 0.65, 'above'),
        ('Low conf rate',   scores['low_conf_rate'],   0.20, 'below'),
    ]
    print("\n=== VALIDATION SUMMARY ===")
    for name, value, threshold, direction in checks:
        passed = (value > threshold) if direction == 'above' \
                 else (value < threshold)
        icon   = "✓" if passed else "✗"
        print(f"  {icon}  {name}: {value:.3f}  (threshold: {threshold})")
    print(
        f"\n  {'READY FOR SEQUENCE MODEL' if all_checks_pass(scores) else 'REFINE PROMPT FIRST'}"
    )

print("Validation gate ready.")

In [ ]:
# 10 Full validation gate

def extract_text_blocks(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return '\n'.join(
            b.get('text', '')
            for b in content
            if isinstance(b, dict) and b.get('type') == 'text'
        ).strip()
    return ''

def extract_tool_calls(content):
    if not isinstance(content, list):
        return []
    return [
        b.get('name', 'unknown')
        for b in content
        if isinstance(b, dict) and b.get('type') == 'tool_use'
    ]

def extract_tool_results(content):
    if not isinstance(content, list):
        return None
    parts = []
    for block in content:
        if not isinstance(block, dict) or block.get('type') != 'tool_result':
            continue
        sub = block.get('content', [])
        if isinstance(sub, str):
            parts.append(sub)
        elif isinstance(sub, list):
            for item in sub:
                if isinstance(item, dict) and item.get('type') == 'text':
                    parts.append(item.get('text', ''))
    return '\n---\n'.join(p for p in parts if p).strip() or None

def get_role_and_content(record):
    if 'message' in record:
        msg = record['message']
        return msg.get('role'), msg.get('content', ''), record.get('timestamp')
    role = record.get('role') or record.get('type')
    return role, record.get('content', ''), record.get('timestamp')

def parse_transcript(path, max_chars=2000):
    rows = []
    prior_tool_result    = None
    prior_assistant_text = None
    prior_user_text      = None
    message_id = 0

    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            role, content, ts = get_role_and_content(record)

            if role == 'user':
                tool_result = extract_tool_results(content)
                if tool_result:
                    prior_tool_result = tool_result[:max_chars]
                user_text = extract_text_blocks(content)
                if user_text:
                    prior_user_text = user_text[:max_chars]
                continue

            if role != 'assistant':
                continue

            text = extract_text_blocks(content)
            if not text:
                continue

            rows.append({
                'message_id':              message_id,
                'content':                 text[:max_chars],
                'has_tool_calls':          bool(extract_tool_calls(content)),
                'tool_names':              extract_tool_calls(content),
                'prior_tool_result':       prior_tool_result,
                'prior_assistant_content': prior_assistant_text,
                'prior_user_content':      prior_user_text,
                'timestamp':               ts,
            })

            prior_assistant_text = text[:max_chars]
            message_id += 1

    return pd.DataFrame(rows)

# --- Run it ---
print("Parsing transcript...")
df = parse_transcript(TRANSCRIPT_PATH)

print(f"Total assistant messages:         {len(df):,}")
print(f"With tool calls:                  {df['has_tool_calls'].sum():,}")
print(f"With prior tool result:           {df['prior_tool_result'].notna().sum():,}")
print(f"With prior assistant context:     {df['prior_assistant_content'].notna().sum():,}")
print(f"\nSample (first 3 rows):")
display(df[['message_id', 'has_tool_calls', 'tool_names', 'content']].head(3))

In [ ]:
# Cell 11 Annotation system prompt

SYSTEM_PROMPT = """You are an expert annotator analyzing reasoning quality in AI assistant conversations.

You will receive a single assistant message along with its context. Classify the message using exactly one of these labels:

GROUNDED     — claim or decision directly supported by evidence in the prior tool result
INFERRED     — reasonable interpretation or extrapolation beyond what tool output explicitly states
ASSERTED     — claim stated as fact with no supporting evidence in context (speculation presented as certain)
CORRECTING   — explicit revision of a prior claim or approach ("I was wrong", "let me revert", "actually...")
ACTION       — message is primarily issuing commands or writing code; reasoning content is minimal
MIXED        — contains a clear mix of 2+ distinct categories above

Rules:
- If prior tool result is "none", GROUNDED is almost never correct
- CORRECTING requires explicit acknowledgment of a previous error, not just changing direction
- ASSERTED is not negative — it describes confidence level, not wrongness
- Prioritize the DOMINANT reasoning type; use MIXED only when truly split

Respond with this exact JSON structure:
{
  "classification": "<one of the six labels>",
  "confidence": <float 0.0–1.0>,
  "evidence_present": <true|false>,
  "evidence_source": "<tool result|prior context|session summary|none>",
  "reasoning": "<one sentence explaining your choice>"
}"""

print("System prompt defined.")
print(f"Prompt length: {len(SYSTEM_PROMPT)} chars")

In [ ]:
# Cell 12 Annotate a sample of 30 messages

SAMPLE_SIZE = 30

sample_df = df.sample(SAMPLE_SIZE, random_state=42).copy()

annotations = []
for i, (_, row) in enumerate(sample_df.iterrows()):
    print(f"Annotating {i+1}/{SAMPLE_SIZE}...", end='\r')
    try:
        ann = annotate_with_retry(
            row['prior_tool_result'],
            row['prior_assistant_content'],
            row['content'],
            SYSTEM_PROMPT
        )
        annotations.append(ann)
    except Exception as e:
        print(f"\n  Error on message {row['message_id']}: {e}")
        annotations.append({
            'classification': 'ERROR',
            'confidence':     0.0,
            'evidence_present': False,
            'evidence_source': 'none',
            'reasoning': str(e)
        })
    time.sleep(0.5)

sample_df = sample_df.reset_index(drop=True)
sample_df['annotation'] = annotations
sample_df['classification'] = sample_df['annotation'].apply(
    lambda a: a.get('classification', 'ERROR')
)
sample_df['confidence'] = sample_df['annotation'].apply(
    lambda a: a.get('confidence', 0.0)
)

print(f"\nDone. Label distribution:")
print(sample_df['classification'].value_counts())
print(f"\nMean confidence: {sample_df['confidence'].mean():.2f}")

In [ ]:
# Cell 13

for _, row in sample_df.sample(5, random_state=7).iterrows():
    print(f"--- Message {row['message_id']} ---")
    print(f"Content:        {row['content'][:120]}")
    print(f"Classification: {row['classification']}")
    print(f"Confidence:     {row['confidence']}")
    print(f"Reasoning:      {row['annotation'].get('reasoning', '')}")
    print()

In [ ]:
# Cell 14

det_rate, det_df = check_determinism(sample_df, SYSTEM_PROMPT, n_runs=3, sample_size=20)

In [ ]:
# Cell

violation_rate, checked_df = run_logic_checks(sample_df)

In [ ]:
# Cell 15 Compute low confidence rate and print the validation summary

low_conf_rate = (sample_df['confidence'] < 0.7).mean()
asserted_recall = 1.0  # placeholder — no ASSERTED in sample to measure against

scores = {
    'determinism':     det_rate,
    'violation_rate':  violation_rate,
    'asserted_recall': asserted_recall,
    'low_conf_rate':   low_conf_rate,
}

print_validation_summary(scores)

In [ ]:
#Cell 16

import os

CHECKPOINT_PATH = '/content/drive/MyDrive/annotations_checkpoint.jsonl'

# Resume from checkpoint if it exists
completed = {}
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r') as f:
        for line in f:
            rec = json.loads(line)
            completed[rec['message_id']] = rec['annotation']
    print(f"Resuming from checkpoint: {len(completed)} already done")

annotations_out = open(CHECKPOINT_PATH, 'a')

full_df = df.copy()
errors  = []

for i, (_, row) in enumerate(full_df.iterrows()):
    mid = row['message_id']

    if mid in completed:
        continue

    if i % 50 == 0:
        print(f"Progress: {i}/{len(full_df)} ({i/len(full_df)*100:.0f}%)...")

    try:
        ann = annotate_with_retry(
            row['prior_tool_result'],
            row['prior_assistant_content'],
            row['content'],
            SYSTEM_PROMPT
        )
    except Exception as e:
        print(f"  Failed message {mid}: {e}")
        ann = {
            'classification':  'ERROR',
            'confidence':       0.0,
            'evidence_present': False,
            'evidence_source':  'none',
            'reasoning':        str(e)
        }
        errors.append(mid)

    completed[mid] = ann
    annotations_out.write(json.dumps({'message_id': mid, 'annotation': ann}) + '\n')
    annotations_out.flush()
    time.sleep(0.5)

annotations_out.close()

# Merge back into dataframe
full_df['annotation']     = full_df['message_id'].map(completed)
full_df['classification'] = full_df['annotation'].apply(
    lambda a: a.get('classification', 'ERROR')
)
full_df['confidence']     = full_df['annotation'].apply(
    lambda a: a.get('confidence', 0.0)
)

print(f"\nDone. {len(full_df)} messages annotated, {len(errors)} errors.")
print("\nLabel distribution:")
print(full_df['classification'].value_counts())
print(f"\nMean confidence: {full_df['confidence'].mean():.2f}")

In [ ]:
# Cell 17 Transition matrix

le = LabelEncoder()
full_df['label_int'] = le.fit_transform(full_df['classification'])
label_names = le.classes_

transitions = np.zeros((len(label_names), len(label_names)), dtype=int)
labels = full_df['classification'].tolist()
for i in range(len(labels) - 1):
    a = list(label_names).index(labels[i])
    b = list(label_names).index(labels[i + 1])
    transitions[a][b] += 1

row_sums = transitions.sum(axis=1, keepdims=True)
trans_prob = np.divide(transitions, row_sums, where=row_sums > 0)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    trans_prob,
    annot=True, fmt='.2f',
    xticklabels=label_names,
    yticklabels=label_names,
    cmap='Blues', ax=ax
)
ax.set_title('Reasoning State Transition Probabilities')
ax.set_xlabel('Next state')
ax.set_ylabel('Current state')
plt.tight_layout()
plt.show()

print("\nRaw transition counts:")
print(pd.DataFrame(transitions, index=label_names, columns=label_names))

In [ ]:
# Cell 18 Reasoning quality timeline

color_map = {
    'GROUNDED':   '#2ecc71',
    'INFERRED':   '#3498db',
    'ACTION':     '#95a5a6',
    'MIXED':      '#f39c12',
    'CORRECTING': '#9b59b6',
    'ASSERTED':   '#e74c3c',
}

fig, ax = plt.subplots(figsize=(18, 4))
for _, row in full_df.iterrows():
    ax.axvline(
        x=row['message_id'],
        color=color_map.get(row['classification'], '#000'),
        alpha=0.6, linewidth=1.2
    )

from matplotlib.patches import Patch
legend = [Patch(color=c, label=l) for l, c in color_map.items()]
ax.legend(handles=legend, loc='upper right', fontsize=9)
ax.set_xlim(0, len(full_df))
ax.set_xlabel('Message sequence')
ax.set_title('Reasoning State Timeline Full Conversation')
ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Cell 19 HHM sequence model to find hidden reasoning quality regimes

from hmmlearn import hmm

sequence = full_df['label_int'].values.reshape(-1, 1)
n_hidden  = 3

model = hmm.CategoricalHMM(n_components=n_hidden, n_iter=100, random_state=42)
model.fit(sequence)

hidden_states = model.predict(sequence)
full_df['hidden_state'] = hidden_states

print("Hidden state distribution:")
print(pd.Series(hidden_states).value_counts().sort_index())

print("\nLabel breakdown per hidden state:")
print(pd.crosstab(full_df['hidden_state'], full_df['classification'], normalize='index').round(2))

In [ ]:
# Cell 20 label and visualization hidden states

state_names = {0: 'Active Reasoning', 1: 'Evidence-Grounded', 2: 'Execution/Drift'}
full_df['state_name'] = full_df['hidden_state'].map(state_names)

state_colors = {
    'Evidence-Grounded': '#2ecc71',
    'Active Reasoning':  '#3498db',
    'Execution/Drift':   '#e74c3c',
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 6), sharex=True)

# Top: hidden states
for _, row in full_df.iterrows():
    ax1.axvline(x=row['message_id'],
                color=state_colors[row['state_name']],
                alpha=0.5, linewidth=1.2)
ax1.set_title('Hidden Reasoning State (HMM)')
ax1.set_yticks([])

# Bottom: ASSERTED and CORRECTING only
for _, row in full_df[full_df['classification'].isin(['ASSERTED', 'CORRECTING'])].iterrows():
    c = '#e74c3c' if row['classification'] == 'ASSERTED' else '#9b59b6'
    ax2.axvline(x=row['message_id'], color=c, alpha=0.8, linewidth=1.5)
ax2.set_title('ASSERTED (red) and CORRECTING (purple) events')
ax2.set_yticks([])
ax2.set_xlabel('Message sequence')

from matplotlib.patches import Patch
ax1.legend(handles=[Patch(color=c, label=l) for l, c in state_colors.items()],
           loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

# Where do ASSERTED messages fall in hidden states?
print("ASSERTED messages by hidden state:")
print(full_df[full_df['classification'] == 'ASSERTED']['state_name'].value_counts())

In [ ]:
# Cell 21 inspect the actual ASSERTED messages

asserted_df = full_df[full_df['classification'] == 'ASSERTED'].copy()

# What came immediately before each ASSERTED message?
asserted_df['prev_classification'] = asserted_df['message_id'].apply(
    lambda mid: full_df.loc[full_df['message_id'] == mid - 1, 'classification'].values[0]
    if mid > 0 else 'START'
)

# What came immediately after?
asserted_df['next_classification'] = asserted_df['message_id'].apply(
    lambda mid: full_df.loc[full_df['message_id'] == mid + 1, 'classification'].values[0]
    if mid < len(full_df) - 1 else 'END'
)

print("What precedes an ASSERTED message:")
print(asserted_df['prev_classification'].value_counts())
print("\nWhat follows an ASSERTED message:")
print(asserted_df['next_classification'].value_counts())

print("\n--- Sample ASSERTED messages ---")
for _, row in asserted_df.sample(8, random_state=42).iterrows():
    print(f"\n[{row['message_id']}] prev={row['prev_classification']} → ASSERTED → next={row['next_classification']}")
    print(f"Content:   {row['content'][:200]}")
    print(f"Reasoning: {row['annotation'].get('reasoning', '')}")

In [ ]:
# Cell 22 extracts the dangerous chains

dangerous = []
labels = full_df['classification'].tolist()
ids    = full_df['message_id'].tolist()

for i in range(1, len(labels) - 1):
    if labels[i] == 'ASSERTED' and labels[i+1] == 'ACTION':
        dangerous.append({
            'asserted_id':  ids[i],
            'prev_label':   labels[i-1],
            'next_label':   labels[i+1],
            'content':      full_df.iloc[i]['content'][:250],
            'reasoning':    full_df.iloc[i]['annotation'].get('reasoning', '')
        })

print(f"ASSERTED → ACTION sequences: {len(dangerous)}")
for d in dangerous:
    print(f"\n[{d['asserted_id']}] {d['prev_label']} → ASSERTED → ACTION")
    print(f"Content:   {d['content']}")
    print(f"Reasoning: {d['reasoning']}")

In [ ]:
# Cell 23 save the final annotated dataset to Drive

output_path = '/content/drive/MyDrive/session-828a165e-annotated.csv'
full_df.drop(columns=['annotation']).to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Rows: {len(full_df)}")

In [ ]:
# Cell 24 Scan for thinking blocks

thinking_count = 0
total_assistant = 0
samples = []

with open(TRANSCRIPT_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue

        role, content, _ = get_role_and_content(record)
        if role != 'assistant':
            continue

        total_assistant += 1

        if not isinstance(content, list):
            continue

        for block in content:
            if isinstance(block, dict) and block.get('type') in ('thinking', 'redacted_thinking'):
                thinking_count += 1
                if len(samples) < 3:
                    samples.append({
                        'type':    block.get('type'),
                        'preview': block.get('thinking', '')[:300] if block.get('thinking') else '[redacted]'
                    })

print(f"Total assistant records:     {total_assistant:,}")
print(f"Records with thinking block: {thinking_count:,}")
print(f"Coverage:                    {thinking_count/total_assistant*100:.1f}%")

if samples:
    print("\n--- Sample thinking blocks ---")
    for s in samples:
        print(f"\nType: {s['type']}")
        print(f"Content: {s['preview']}")
else:
    print("\nNo thinking blocks found.")

In [ ]:
# Cell 25 Re-parse to capture thinking block flag per message

THINKING_TYPES = ('thinking', 'redacted_thinking')

def parse_transcript_with_thinking(path, max_chars=2000):
    rows = []
    prior_tool_result    = None
    prior_assistant_text = None
    prior_user_text      = None
    message_id = 0

    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            role, content, ts = get_role_and_content(record)

            if role == 'user':
                tool_result = extract_tool_results(content)
                if tool_result:
                    prior_tool_result = tool_result[:max_chars]
                user_text = extract_text_blocks(content)
                if user_text:
                    prior_user_text = user_text[:max_chars]
                continue

            if role != 'assistant':
                continue

            had_thinking = isinstance(content, list) and any(
                isinstance(b, dict) and b.get('type') in THINKING_TYPES
                for b in content
            )

            text = extract_text_blocks(content)
            if not text:
                continue

            rows.append({
                'message_id':              message_id,
                'content':                 text[:max_chars],
                'had_thinking':            had_thinking,
                'prior_tool_result':       prior_tool_result,
                'prior_assistant_content': prior_assistant_text,
                'prior_user_content':      prior_user_text,
                'timestamp':               ts,
            })

            prior_assistant_text = text[:max_chars]
            message_id += 1

    return pd.DataFrame(rows)

print("Re-parsing...")
df2 = parse_transcript_with_thinking(TRANSCRIPT_PATH)
print(f"Messages:              {len(df2):,}")
print(f"With thinking block:   {df2['had_thinking'].sum():,} ({df2['had_thinking'].mean()*100:.1f}%)")
print(f"Without thinking block:{(~df2['had_thinking']).sum():,}")

In [ ]:
# Cell 26 Raw message, first 20 lines, and raw structure diagnostic

print("=== FIRST 20 RAW LINES ===\n")
with open(TRANSCRIPT_PATH, 'r') as f:
    for i, line in enumerate(f):
        if i >= 20:
            break
        print(f"Line {i+1}: {line[:300]}")

print("\n=== FIRST RECORD (full) ===\n")
with open(TRANSCRIPT_PATH, 'r') as f:
    first = json.loads(f.readline())
print(json.dumps(first, indent=2))

thinking_records = []
record_types = {}

with open(TRANSCRIPT_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue

        # Count all top-level record types
        rtype = record.get('type', record.get('role', 'unknown'))
        record_types[rtype] = record_types.get(rtype, 0) + 1

        # Capture first 3 records that mention "thinking" anywhere
        if len(thinking_records) < 3:
            record_str = json.dumps(record)
            if 'thinking' in record_str.lower():
                thinking_records.append(record)

print("Top-level record type distribution:")
for k, v in sorted(record_types.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,}")

print(f"\nRecords mentioning 'thinking': {len(thinking_records)} samples found")
for i, r in enumerate(thinking_records):
    print(f"\n--- Sample {i+1} ---")
    print(json.dumps(r, indent=2)[:800])

In [ ]:
# Cell 27 Re-parse grouping by message ID

from collections import defaultdict

message_groups = defaultdict(list)

with open(TRANSCRIPT_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue
        if record.get('type') != 'assistant':
            continue
        msg_id = record.get('message', {}).get('id')
        if msg_id:
            message_groups[msg_id].append(record)

rows = []
prior_tool_result    = None
prior_assistant_text = None
message_id = 0

for msg_id, records in message_groups.items():
    had_thinking = False
    text_content = None
    tool_names   = []

    for r in records:
        content = r.get('message', {}).get('content', [])
        if not isinstance(content, list):
            continue
        for block in content:
            if not isinstance(block, dict):
                continue
            btype = block.get('type')
            if btype in ('thinking', 'redacted_thinking'):
                had_thinking = True
            elif btype == 'text':
                text_content = block.get('text', '')
            elif btype == 'tool_use':
                tool_names.append(block.get('name', 'unknown'))

    if not text_content:
        continue

    rows.append({
        'message_id':   message_id,
        'msg_api_id':   msg_id,
        'content':      text_content[:2000],
        'had_thinking': had_thinking,
        'tool_names':   tool_names,
    })
    message_id += 1

df3 = pd.DataFrame(rows)

print(f"Messages with text:          {len(df3):,}")
print(f"With thinking block:         {df3['had_thinking'].sum():,} ({df3['had_thinking'].mean()*100:.1f}%)")
print(f"Without thinking block:      {(~df3['had_thinking']).sum():,}")

print("\nSample — first 5 rows with thinking flag:")
display(df3[['message_id', 'had_thinking', 'content']].head(5))

In [ ]:
#Cell 28 Merge corrected had_thinking and has_tool_calls into full_df

full_df = full_df.merge(
    df3[['message_id', 'had_thinking', 'tool_names']],
    on='message_id',
    how='left',
    suffixes=('_old', '')
)

print("ASSERTED rate by thinking block presence:")
asserted_by_thinking = full_df.groupby('had_thinking')['classification'].apply(
    lambda x: (x == 'ASSERTED').mean()
).rename('asserted_rate').round(4)
print(asserted_by_thinking)

print("\nFull label distribution by thinking block:")
print(pd.crosstab(
    full_df['had_thinking'],
    full_df['classification'],
    normalize='index'
).round(3))

print("\nASSERTED message counts:")
print(full_df[full_df['classification'] == 'ASSERTED']['had_thinking'].value_counts())

In [ ]:
# Cell 29 Chi-square test

from scipy import stats
import numpy as np

# Recompute from full_df (requires Cell 28 merge — had_thinking column present)
with_t    = full_df[full_df['had_thinking'] == True]
without_t = full_df[full_df['had_thinking'] == False]

n_with,    n_without    = len(with_t),    len(without_t)
a_with,    a_without    = (with_t['classification'] == 'ASSERTED').sum(), \
                          (without_t['classification'] == 'ASSERTED').sum()

table = np.array([
    [a_with,    n_with    - a_with],
    [a_without, n_without - a_without]
])

print("Contingency table:")
print(f"                  ASSERTED   NOT ASSERTED   TOTAL")
print(f"With thinking:    {a_with:6d}   {n_with-a_with:10d}   {n_with}")
print(f"Without thinking: {a_without:6d}   {n_without-a_without:10d}   {n_without}")

# Pearson chi-square (no correction)
chi2, p, dof, _ = stats.chi2_contingency(table, correction=False)
print(f"\nChi-square (Pearson):    χ²={chi2:.4f}, p={p:.4f}, df={dof}")

# Yates' correction (more conservative for 2×2)
chi2_y, p_y, _, _ = stats.chi2_contingency(table, correction=True)
print(f"Chi-square (Yates):      χ²={chi2_y:.4f}, p={p_y:.4f}")

# Fisher's exact (best choice — one cell has only 6 observations)
odds_ratio, p_fisher = stats.fisher_exact(table, alternative='two-sided')
print(f"Fisher's exact:          OR={odds_ratio:.4f}, p={p_fisher:.4f}")

# Effect size: Cramér's V
cramer_v = np.sqrt(chi2 / table.sum())
size_label = 'small' if cramer_v < 0.1 else 'medium' if cramer_v < 0.3 else 'large'
print(f"Cramér's V:              {cramer_v:.4f} ({size_label} effect)")

alpha = 0.05
sig   = p_fisher < alpha
print(f"\n{'Statistically significant (p < 0.05)' if sig else 'NOT statistically significant (p ≥ 0.05)'}")
print(f"The {a_with/n_with*100:.2f}% vs {a_without/n_without*100:.2f}% difference")
print(f"{'IS' if sig else 'is NOT'} likely due to a real association — Extended Thinking {'is associated with' if sig else 'does not reliably predict'} higher ASSERTED rate.")
print("Note: correlation likely confounded — harder problems trigger both thinking and more drift.")

In [ ]:
#Cell 30 Sentence embedding + kmeans on 31 ASSERTED mesages

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

asserted_df = full_df[full_df['classification'] == 'ASSERTED'].reset_index(drop=True)
print(f"ASSERTED messages to cluster: {len(asserted_df)}")

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Encoding...")
embeddings = model.encode(asserted_df['content'].tolist(), show_progress_bar=True)
print(f"Embedding shape: {embeddings.shape}")

# Elbow — try k=2..6
inertias = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(embeddings)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 3))
plt.plot(range(2, 7), inertias, 'bo-')
plt.xlabel('k'); plt.ylabel('Inertia')
plt.title('Elbow — ASSERTED clusters')
plt.tight_layout(); plt.show()

# Fit k=4 (our manual taxonomy has 4 types)
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
asserted_df = asserted_df.copy()
asserted_df['cluster'] = km4.fit_predict(embeddings)

# PCA scatter
pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings)
asserted_df['pca_x'], asserted_df['pca_y'] = coords[:, 0], coords[:, 1]

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
plt.figure(figsize=(8, 5))
for c in range(4):
    mask = asserted_df['cluster'] == c
    plt.scatter(asserted_df.loc[mask, 'pca_x'], asserted_df.loc[mask, 'pca_y'],
                color=colors[c], label=f'Cluster {c}', s=90, edgecolors='white', linewidths=0.4)
    for _, row in asserted_df[mask].iterrows():
        plt.annotate(str(int(row['message_id'])), (row['pca_x'], row['pca_y']),
                     fontsize=7, color='white', alpha=0.8)
plt.title('KMeans k=4 — ASSERTED messages (PCA)')
plt.legend(); plt.tight_layout(); plt.show()

# Manual taxonomy reference for comparison
manual_taxonomy = {
    'pre_diagnosis':    [590, 601, 732],
    'context_recon':    [1, 267, 673],
    'evidence_contra':  [556],
    'general_knowledge':[790, 1059, 1136, 1189, 1242],
}

print("\nCluster members (compare to manual taxonomy):")
for c in range(4):
    members = asserted_df[asserted_df['cluster'] == c].sort_values('message_id')
    ids = sorted(members['message_id'].astype(int).tolist())
    print(f"\nCluster {c} — {len(members)} messages — IDs: {ids}")
    for _, row in members.iterrows():
        print(f"  [{int(row['message_id']):4d}] {row['content'][:120]}")

In [ ]:
# Cell 31 Drift Velocity

import numpy as np
import matplotlib.pyplot as plt

labels      = full_df['classification'].tolist()
message_ids = full_df['message_id'].tolist()

def steps_since_last_grounded(labels, idx):
    for i in range(idx - 1, -1, -1):
        if labels[i] == 'GROUNDED':
            return idx - i - 1
    return idx  # no prior GROUNDED — count from start

asserted_idx = [i for i, l in enumerate(labels) if l == 'ASSERTED']
drift_records = [{'message_id': message_ids[i],
                  'drift_distance': steps_since_last_grounded(labels, i)}
                 for i in asserted_idx]
drift_df = pd.DataFrame(drift_records)

print("Drift velocity — messages since last GROUNDED before each ASSERTED event")
print(drift_df['drift_distance'].describe().to_string())
print(f"\nMedian:    {drift_df['drift_distance'].median():.1f}")
print(f"Mean:      {drift_df['drift_distance'].mean():.1f}")
print(f"90th pct:  {drift_df['drift_distance'].quantile(0.90):.0f}")
print(f"Max:       {drift_df['drift_distance'].max()}")

max_bin = int(drift_df['drift_distance'].max()) + 2
plt.figure(figsize=(9, 4))
plt.hist(drift_df['drift_distance'], bins=range(0, max_bin), color='#e74c3c',
         edgecolor='#1a1a1a', alpha=0.85)
plt.axvline(drift_df['drift_distance'].median(), color='#f39c12', linestyle='--',
            label=f"Median: {drift_df['drift_distance'].median():.0f}")
plt.axvline(drift_df['drift_distance'].quantile(0.90), color='white', linestyle=':',
            label=f"90th pct: {drift_df['drift_distance'].quantile(0.90):.0f}")
plt.xlabel('Messages since last GROUNDED')
plt.ylabel('ASSERTED events')
plt.title('Drift Velocity — Distance from Evidence to Assertion (speaking turns only)')
plt.legend(); plt.tight_layout(); plt.show()

print("\nPer-event breakdown (sorted by distance):")
for _, row in drift_df.sort_values('drift_distance', ascending=False).iterrows():
    print(f"  msg {int(row['message_id']):4d}: {int(row['drift_distance'])} steps from last GROUNDED")

In [ ]:
#Cell 32 Silent action integration + full-sequence transtion matrix + drift velocity comparison

import json, numpy as np, matplotlib.pyplot as plt
from collections import defaultdict, Counter

TRANSCRIPT_PATH = '/content/drive/MyDrive/session-828a165e-full-transcript-20260715.jsonl'
THINKING_TYPES  = ('thinking', 'redacted_thinking')

# ── 1. Parse all turns from JSONL, grouped by message.id ────────────────
message_groups = defaultdict(list)
with open(TRANSCRIPT_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: record = json.loads(line)
        except: continue
        if record.get('type') != 'assistant': continue
        mid = record.get('message', {}).get('id')
        if mid: message_groups[mid].append(record)

all_turns, text_seq_id = [], 0
for msg_id, records in message_groups.items():
    had_thinking, text_content, tool_names = False, None, []
    for r in records:
        content = r.get('message', {}).get('content', [])
        if not isinstance(content, list): continue
        for block in content:
            if not isinstance(block, dict): continue
            bt = block.get('type')
            if bt in THINKING_TYPES:          had_thinking = True
            elif bt == 'text':                text_content = block.get('text', '')
            elif bt == 'tool_use':            tool_names.append(block.get('name', 'unknown'))
    is_silent = (text_content is None) and bool(tool_names)
    turn = {'msg_api_id': msg_id, 'tool_names': tool_names,
            'had_thinking': had_thinking, 'is_silent': is_silent,
            'seq_id': None}
    if not is_silent:
        turn['seq_id'] = text_seq_id
        text_seq_id += 1
    all_turns.append(turn)

text_turns   = [t for t in all_turns if not t['is_silent']]
silent_turns = [t for t in all_turns if t['is_silent']]
print(f"Total turns:   {len(all_turns)}")
print(f"  TEXT:         {len(text_turns)}")
print(f"  SILENT:       {len(silent_turns)}")

tool_counts = Counter(n for t in silent_turns for n in t['tool_names'])
print(f"\nTools in silent turns:")
for tool, cnt in tool_counts.most_common():
    print(f"  {tool}: {cnt}")

# ── 2. Assign labels ─────────────────────────────────────────────────────
label_by_seq = dict(zip(full_df['message_id'], full_df['classification']))

def silent_label(tool_names):
    tools = set(tool_names)
    if 'Write' in tools or 'Edit' in tools: return 'SILENT_WRITE'
    if 'Bash'  in tools:                    return 'SILENT_BASH'
    if 'Read'  in tools:                    return 'SILENT_READ'
    return 'SILENT_ACTION'

full_sequence = []
for t in all_turns:
    if t['is_silent']:
        full_sequence.append(silent_label(t['tool_names']))
    else:
        full_sequence.append(label_by_seq.get(t['seq_id'], 'UNKNOWN'))

counts = Counter(full_sequence)
print(f"\nFull 2,262-turn label distribution:")
for lbl, cnt in counts.most_common():
    print(f"  {lbl:20s}: {cnt:5d}  ({cnt/len(full_sequence)*100:.1f}%)")

# ── 3. Transition matrix — ASSERTED row in full sequence ─────────────────
trans = defaultdict(lambda: defaultdict(int))
for a, b in zip(full_sequence[:-1], full_sequence[1:]):
    trans[a][b] += 1

print("\nASSERTED transitions (full 2,262-turn sequence):")
a_total = sum(trans['ASSERTED'].values())
for tgt, cnt in sorted(trans['ASSERTED'].items(), key=lambda x: -x[1]):
    print(f"  ASSERTED → {tgt:20s}: {cnt:3d}  ({cnt/a_total*100:.1f}%)")

# ── 4. Drift velocity on full sequence — compare with speaking-only ───────
def drift_full(seq, idx):
    for i in range(idx - 1, -1, -1):
        if seq[i] == 'GROUNDED':
            return idx - i - 1
    return idx

asserted_idx_full = [i for i, l in enumerate(full_sequence) if l == 'ASSERTED']
full_distances    = [drift_full(full_sequence, i) for i in asserted_idx_full]

# speaking-turns-only distances from Cell 31
speaking_distances = drift_df['drift_distance'].tolist() if 'drift_df' in dir() else []

print(f"\nDrift velocity comparison:")
print(f"  {'Metric':<20} {'Speaking only':>16} {'Full sequence':>14}")
print(f"  {'-'*50}")
metrics = [('Median', np.median), ('Mean', np.mean),
           ('90th pct', lambda x: np.percentile(x, 90)), ('Max', max)]
for name, fn in metrics:
    sp = f"{fn(speaking_distances):.1f}" if speaking_distances else 'n/a'
    print(f"  {name:<20} {sp:>16} {fn(full_distances):>14.1f}")

# Histogram overlay
plt.figure(figsize=(9, 4))
if speaking_distances:
    plt.hist(speaking_distances, bins=range(0, max(max(speaking_distances), max(full_distances)) + 2),
             color='#3498db', alpha=0.6, label='Speaking turns only', edgecolor='#1a1a1a')
plt.hist(full_distances, bins=range(0, max(full_distances) + 2),
         color='#e74c3c', alpha=0.6, label='Full 2,262-turn sequence', edgecolor='#1a1a1a')
plt.axvline(np.median(full_distances), color='#f39c12', linestyle='--',
            label=f"Full median: {np.median(full_distances):.0f}")
plt.xlabel('Steps since last GROUNDED')
plt.ylabel('ASSERTED events')
plt.title('Drift Velocity — Speaking Turns vs Full Sequence')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Cell 33 Evidence gap analysis; for each of the 31 asserted,
# ask Gemini to identify the specific gap between what the prior tool returned and what the speaking turn claimed

client = gemini_client
MODEL_NAME = WORKING_MODEL

import json, time, pandas as pd
from collections import Counter

EVIDENCE_GAP_PROMPT = """You are analyzing one assistant message from a Claude Code session.

PRIOR TOOL RESULT (what the tool actually returned before this message):
{prior_tool_result}

SPEAKING TURN (what the assistant said — already classified as ASSERTED):
{content}

Task: identify the EVIDENCE GAP.
List each specific claim in the speaking turn that CANNOT be traced to anything
in the prior tool result. For each unsupported claim, quote it exactly and explain
what would need to have appeared in the tool result to support it.
If ALL claims are traceable to the tool result, say so.

Respond ONLY in this JSON format:
{{
  "unsupported_claims": [
    {{
      "claim": "exact quote from speaking turn",
      "gap": "what is missing from tool result to support this claim"
    }}
  ],
  "gap_summary": "one sentence describing the overall nature of the gap",
  "confidence": 0.0
}}"""

# --- identify message_id column ---
print("full_df columns:", list(full_df.columns))
print("full_df shape:", full_df.shape)

# Adjust this if your column is named differently (uuid, msg_id, etc.)
ID_COL = 'message_id' if 'message_id' in full_df.columns else full_df.columns[0]
print(f"Using '{ID_COL}' as message ID column\n")

asserted_df = full_df[full_df['classification'] == 'ASSERTED'].copy().reset_index(drop=True)
print(f"Analyzing {len(asserted_df)} ASSERTED turns...\n")

gap_results = []

for i, row in asserted_df.iterrows():
    msg_id   = str(row.get(ID_COL, f'row_{i}'))
    content  = str(row.get('content', ''))[:2000]
    prior    = str(row.get('prior_tool_result', ''))[:2000]

    prompt = EVIDENCE_GAP_PROMPT.format(
        prior_tool_result=prior if prior.strip() else '[No prior tool result captured]',
        content=content
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config={'temperature': 0}
        )
        raw = response.text.strip()
        if raw.startswith('```'):
            raw = '\n'.join(raw.split('\n')[1:-1])
        result = json.loads(raw)
    except Exception as e:
        result = {
            'unsupported_claims': [],
            'gap_summary': f'ERROR: {e}',
            'confidence': 0.0
        }

    result['message_id']       = msg_id
    result['content_preview']  = content[:150]
    result['prior_preview']    = prior[:150]
    gap_results.append(result)

    n = len(result.get('unsupported_claims', []))
    print(f"[{i+1:02d}/31] {msg_id[:24]}  →  {n} unsupported claim(s)  |  {result.get('gap_summary','')[:80]}")
    time.sleep(0.5)

# --- display ---
print("\n" + "="*60)
print("EVIDENCE GAP ANALYSIS — DETAIL")
print("="*60)
for r in gap_results:
    claims = r.get('unsupported_claims', [])
    if not claims:
        continue
    print(f"\n[{r['message_id']}]")
    print(f"  Summary : {r.get('gap_summary','')}")
    for c in claims:
        print(f"  CLAIM   : \"{c.get('claim','')}\"")
        print(f"  GAP     : {c.get('gap','')}")

# --- summary stats ---
total_gaps   = sum(len(r.get('unsupported_claims', [])) for r in gap_results)
turns_w_gaps = sum(1 for r in gap_results if r.get('unsupported_claims'))
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"  Turns with ≥1 unsupported claim : {turns_w_gaps}/31")
print(f"  Total unsupported claims         : {total_gaps}")
print(f"  Avg per ASSERTED turn            : {total_gaps/len(gap_results):.1f}")

# --- save ---
gap_df = pd.DataFrame([{
    'message_id':          r['message_id'],
    'gap_summary':         r.get('gap_summary', ''),
    'n_unsupported':       len(r.get('unsupported_claims', [])),
    'unsupported_claims':  json.dumps(r.get('unsupported_claims', [])),
    'confidence':          r.get('confidence', 0.0),
    'prior_truncated':     len(r.get('prior_preview', '')) >= 149  # flag truncation
} for r in gap_results])

out_path = '/content/drive/MyDrive/session-828a165e-evidence-gaps.csv'
gap_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

# flag truncation risk
truncated = gap_df['prior_truncated'].sum()
if truncated:
    print(f"⚠ {truncated} turns had prior_tool_result at or near 2,000-char truncation limit.")
    print("  Some GROUNDED evidence may have been cut off — treat those gaps with lower confidence.")

In [ ]:
# Cell 34 Thinking block analysis, for each of the 31 ASSERTED turns, retrieve the thinking block from the source JSONL and
# ask Gemini whether the model was reasoning from the tool result or from prior frame/assertion already in context

import json, time, pandas as pd
from collections import Counter

client = gemini_client
MODEL_NAME = WORKING_MODEL

THINKING_PROMPT = """You are analyzing the internal reasoning of an AI model (its hidden thinking block)
immediately before it produced a response that has been classified as ASSERTED —
a claim not traceable to prior tool output.

PRIOR TOOL RESULT (what the tool returned before this turn):
{prior_tool_result}

MODEL'S INTERNAL THINKING (hidden scratchpad, not shown to the user):
{thinking}

MODEL'S ACTUAL RESPONSE (the ASSERTED turn):
{content}

Task: determine what the thinking block is primarily reasoning FROM.

EVIDENCE-BASED  — thinking primarily references and reasons from the prior tool result
FRAME-BASED     — thinking primarily references a prior claim, assumption, or hypothesis
                  already stated earlier in the conversation, NOT the current tool result
MIXED           — roughly equal reasoning from both sources

Respond ONLY in this JSON format:
{{
  "primary_basis": "EVIDENCE-BASED | FRAME-BASED | MIXED",
  "evidence_references": ["specific items from tool result the thinking cites"],
  "frame_references": ["specific prior claims or assumptions the thinking builds on"],
  "key_reasoning": "one sentence: what drove the thinking before the assertion",
  "confidence": 0.0
}}"""

# ── Step 1: single pass through JSONL ──────────────────────
# Build simultaneously:
#   ordered_text_uuids — UUIDs of messages with text blocks, in JSONL order
#   uuid_to_thinking   — UUID → thinking text
#   uuid_blocks        — UUID → all content blocks (to detect whether it has text)

print("Parsing source JSONL to build turn_number → UUID mapping...")
uuid_blocks      = {}   # UUID → list of content block dicts
uuid_first_seen  = []   # UUIDs in first-seen order (preserves JSONL turn order)
uuid_to_thinking = {}   # UUID → thinking text

with open(TRANSCRIPT_PATH, 'r') as f:
    for line in f:
        try:
            record = json.loads(line)
        except:
            continue

        if record.get('type') != 'assistant':
            continue

        msg = record.get('message', {})
        if not isinstance(msg, dict):
            continue

        uuid = msg.get('id', '')
        if not uuid:
            continue

        if uuid not in uuid_blocks:
            uuid_blocks[uuid] = []
            uuid_first_seen.append(uuid)

        for block in msg.get('content', []):
            if not isinstance(block, dict):
                continue
            uuid_blocks[uuid].append(block)

            if block.get('type') == 'thinking':
                thinking_text = block.get('thinking', '').strip()
                if thinking_text and uuid not in uuid_to_thinking:
                    uuid_to_thinking[uuid] = thinking_text

# ── Step 2: build turn_number → UUID mapping (1-based) ─────
# Only text-producing turns get a turn number (same logic as Cell 10)
ordered_text_uuids = []
for uuid in uuid_first_seen:
    has_text = any(
        b.get('type') == 'text' and b.get('text', '').strip()
        for b in uuid_blocks[uuid]
    )
    if has_text:
        ordered_text_uuids.append(uuid)

turn_to_uuid = {i + 1: uuid for i, uuid in enumerate(ordered_text_uuids)}

print(f"  Speaking turns in JSONL : {len(ordered_text_uuids)}")
print(f"  Speaking turns in full_df: {len(full_df)}")
print(f"  Thinking blocks found   : {len(uuid_to_thinking)}")

# ── Step 3: sanity check — turn 1 content should match full_df row 0 ─
turn1_uuid = turn_to_uuid.get(1, '')
turn1_text_jsonl = ''
for block in uuid_blocks.get(turn1_uuid, []):
    if block.get('type') == 'text':
        turn1_text_jsonl = block.get('text', '')[:80]
        break
turn1_text_df = str(full_df.iloc[0]['content'])[:80]
match = turn1_text_jsonl.strip()[:60] == turn1_text_df.strip()[:60]
print(f"\nSanity check — turn 1:")
print(f"  JSONL text  : {turn1_text_jsonl}")
print(f"  full_df text: {turn1_text_df}")
print(f"  Match       : {'YES' if match else 'NO — check turn numbering'}")

# ── Step 4: retrieve thinking blocks for 31 ASSERTED turns ─
asserted_df = full_df[full_df['classification'] == 'ASSERTED'].copy().reset_index(drop=True)

found_count, missing_turns = 0, []
for _, row in asserted_df.iterrows():
    turn_num = int(row['message_id'])
    uuid = turn_to_uuid.get(turn_num, '')
    if uuid and uuid in uuid_to_thinking:
        found_count += 1
    else:
        missing_turns.append(turn_num)

print(f"\nThinking blocks matched to ASSERTED turns: {found_count}/31")
if missing_turns:
    print(f"  No thinking block for turns: {missing_turns}")

# ── Step 5: Gemini analysis ────────────────────────────────
print("\nRunning Gemini analysis...\n")
thinking_results = []

for i, row in asserted_df.iterrows():
    turn_num = int(row['message_id'])
    uuid     = turn_to_uuid.get(turn_num, '')
    content  = str(row.get('content', ''))[:2000]
    prior    = str(row.get('prior_tool_result', ''))[:2000]
    thinking = uuid_to_thinking.get(uuid, '')

    if not thinking:
        thinking_results.append({
            'message_id':          turn_num,
            'uuid':                uuid,
            'had_thinking_block':  False,
            'primary_basis':       'NO_THINKING_BLOCK',
            'evidence_references': [],
            'frame_references':    [],
            'key_reasoning':       'No thinking block found for this turn',
            'confidence':          0.0
        })
        print(f"[{i+1:02d}/31] turn {turn_num:4d}  →  NO_THINKING_BLOCK")
        continue

    prompt = THINKING_PROMPT.format(
        prior_tool_result=prior if prior.strip() else '[No prior tool result captured]',
        thinking=thinking[:3000],
        content=content
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config={'temperature': 0}
        )
        raw = response.text.strip()
        if raw.startswith('```'):
            raw = '\n'.join(raw.split('\n')[1:-1])
        result = json.loads(raw)
    except Exception as e:
        result = {
            'primary_basis':       f'ERROR: {e}',
            'evidence_references': [],
            'frame_references':    [],
            'key_reasoning':       '',
            'confidence':          0.0
        }

    result['message_id']         = turn_num
    result['uuid']               = uuid
    result['had_thinking_block'] = True
    result['thinking_preview']   = thinking[:200]
    thinking_results.append(result)

    print(f"[{i+1:02d}/31] turn {turn_num:4d}  →  {result.get('primary_basis','?'):14s}  |  {result.get('key_reasoning','')[:65]}")
    time.sleep(0.5)

# ── Step 6: summary ────────────────────────────────────────
print(f"\n{'='*60}")
print("THINKING BLOCK ANALYSIS — SUMMARY")
print("="*60)

has_thinking = [r for r in thinking_results if r.get('had_thinking_block')]
basis_counts = Counter(r['primary_basis'] for r in has_thinking)

for basis, count in basis_counts.most_common():
    pct = count / len(has_thinking) * 100 if has_thinking else 0
    print(f"  {basis:20s}: {count:2d}  ({pct:.1f}%)")
print(f"\n  Based on {len(has_thinking)} turns with thinking blocks "
      f"({len(thinking_results)-len(has_thinking)} had no thinking block)")

frame_n    = basis_counts.get('FRAME-BASED', 0)
evidence_n = basis_counts.get('EVIDENCE-BASED', 0)
mixed_n    = basis_counts.get('MIXED', 0)
total_w    = len(has_thinking) or 1

print(f"\n{'='*60}")
print("KEY FINDING")
print("="*60)
print(f"  Reasoning from prior frame/assertion : {frame_n}/{total_w}  ({frame_n/total_w*100:.1f}%)")
print(f"  Reasoning from tool evidence         : {evidence_n}/{total_w}  ({evidence_n/total_w*100:.1f}%)")
print(f"  Mixed                                : {mixed_n}/{total_w}  ({mixed_n/total_w*100:.1f}%)")

if frame_n > evidence_n:
    print("\n→ SUPPORTS the tool-result boundary claim:")
    print("  Thinking blocks for ASSERTED turns were primarily reasoning from")
    print("  prior frames, not from current tool output. Drift occurred at the")
    print("  tool-result → interpretation boundary.")
elif evidence_n > frame_n:
    print("\n→ CHALLENGES the tool-result boundary claim:")
    print("  Thinking blocks were primarily reasoning from tool evidence.")
    print("  Drift may originate elsewhere in the sequence.")
else:
    print("\n→ INCONCLUSIVE: mixed evidence on boundary location.")

# ── Step 7: save ───────────────────────────────────────────
thinking_df = pd.DataFrame([{
    'message_id':         r['message_id'],
    'uuid':               r.get('uuid', ''),
    'had_thinking_block': r.get('had_thinking_block', False),
    'primary_basis':      r.get('primary_basis', ''),
    'key_reasoning':      r.get('key_reasoning', ''),
    'n_frame_refs':       len(r.get('frame_references', [])),
    'n_evidence_refs':    len(r.get('evidence_references', [])),
    'confidence':         r.get('confidence', 0.0)
} for r in thinking_results])

out_path = '/content/drive/MyDrive/session-828a165e-thinking-analysis.csv'
thinking_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

In [ ]:
# DIAGNOSTIC 34b — Find actual thinking block structure
# We know turn_to_uuid is correct (sanity check passed).
# Now find a turn known to have had_thinking=True and inspect ALL
# JSONL records sharing its message.id — regardless of record type.

turns_with_thinking = full_df[full_df['had_thinking'] == True]
sample_turn_num = int(turns_with_thinking.iloc[0]['message_id'])
sample_uuid = turn_to_uuid.get(sample_turn_num, '')
print(f"Inspecting turn {sample_turn_num} (uuid: {sample_uuid})")
print(f"Content: {str(turns_with_thinking.iloc[0]['content'])[:80]}\n")

# Scan ALL records — don't filter by type — matching this message.id
print("All JSONL records sharing this message.id:")
with open(TRANSCRIPT_PATH, 'r') as f:
    for i, line in enumerate(f):
        try:
            record = json.loads(line)
        except:
            continue
        msg = record.get('message', {})
        if not isinstance(msg, dict) or msg.get('id') != sample_uuid:
            continue
        print(f"\n  Line {i+1}:")
        print(f"    record.type     : {record.get('type')}")
        print(f"    message keys    : {list(msg.keys())}")
        content = msg.get('content', 'MISSING')
        if isinstance(content, list):
            print(f"    content length  : {len(content)}")
            for j, b in enumerate(content):
                if isinstance(b, dict):
                    print(f"    content[{j}].type: {b.get('type')}  keys: {list(b.keys())}")
                    if b.get('type') == 'thinking':
                        print(f"    thinking[:80]   : {b.get('thinking','')[:80]}")
        elif isinstance(content, str):
            print(f"    content (str)   : {content[:80]}")
        else:
            print(f"    content         : {content}")
        if 'thinking' in msg:
            print(f"    message.thinking: {str(msg['thinking'])[:80]}")

# Also show all unique top-level type values
print("\n\nAll record type values in JSONL (top-level):")
from collections import Counter
type_counts = Counter()
with open(TRANSCRIPT_PATH, 'r') as f:
    for line in f:
        try:
            type_counts[json.loads(line).get('type','MISSING')] += 1
        except:
            pass
for t, c in type_counts.most_common():
    print(f"  {t:30s}: {c}")

In [ ]:
# DIAGNOSTIC 34c — find first JSONL line containing "thinking" and show its structure
with open(TRANSCRIPT_PATH, 'r') as f:
    for i, line in enumerate(f):
        if '"thinking"' not in line:
            continue
        print(f"Line {i+1} raw (first 400 chars):\n{line[:400]}\n")
        record = json.loads(line)
        msg = record.get('message', {})
        print(f"record.type     : {record.get('type')}")
        print(f"message.type    : {msg.get('type', 'NOT FOUND')}")
        content = msg.get('content', 'MISSING')
        print(f"content Python type: {type(content).__name__}")
        print(f"content preview : {str(content)[:300]}")
        if 'thinking' in msg:
            print(f"message.thinking: {str(msg['thinking'])[:150]}")

In [ ]:
# Cell 35: SILENT_BASH Origin Split
# For each of the 692 SILENT_BASH turns, classifies the immediately
# preceding user record as:
#   AUTONOMOUS    — preceded by a tool_result user record
#                   (Claude continuing its own investigation after a tool call)
#   USER_PROMPTED — preceded by a human text user record
#                   (User asked Claude to run something; Claude ran without text)
#   AMBIGUOUS     — mixed, missing, or edge-case
#
# Also checks the ASSERTED → SILENT_BASH chains from Cell 32 specifically,
# since those are the ones that underlie Finding One's 61.3% claim.


import json
from collections import OrderedDict, Counter

# 1. Parse JSONL into ordered turns
# Assistant: group multiple records by rec['message']['id']
# User: each record is one turn, identified by rec['uuid']

turns_by_id = OrderedDict()

with open(TRANSCRIPT_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        rec_type = rec.get('type')
        if rec_type not in ('assistant', 'user'):
            continue

        message = rec.get('message', {})

        if rec_type == 'assistant':
            turn_id = message.get('id')
            if not turn_id:
                continue
            if turn_id not in turns_by_id:
                turns_by_id[turn_id] = {'role': 'assistant', 'blocks': []}
            content = message.get('content', [])
            if isinstance(content, list):
                turns_by_id[turn_id]['blocks'].extend(content)

        else:
            turn_id = rec.get('uuid')
            if not turn_id:
                continue
            if turn_id not in turns_by_id:
                content_raw = message.get('content')
                turns_by_id[turn_id] = {'role': 'user', 'content_raw': content_raw}

print("Total turns: " + str(len(turns_by_id)))

# 2. Characterize each turn
def characterize(entry):
    role = entry['role']
    if role == 'assistant':
        blocks = entry.get('blocks', [])
        block_types = [b.get('type', '') for b in blocks if isinstance(b, dict)]
        tool_names  = [b.get('name', '') for b in blocks
                       if isinstance(b, dict) and b.get('type') == 'tool_use']
        return {
            'role':         'assistant',
            'has_text':     any(t == 'text'     for t in block_types),
            'has_tool_use': any(t == 'tool_use' for t in block_types),
            'tool_names':   tool_names,
        }
    else:
        content = entry.get('content_raw')
        if isinstance(content, str):
            return {'role': 'user', 'is_human': True,  'is_tool_result': False}
        elif isinstance(content, list):
            types = [b.get('type', '') for b in content if isinstance(b, dict)]
            is_tr = any(t == 'tool_result' for t in types)
            return {'role': 'user', 'is_human': not is_tr, 'is_tool_result': is_tr}
        else:
            return {'role': 'user', 'is_human': False, 'is_tool_result': False}

messages = [(tid, characterize(entry)) for tid, entry in turns_by_id.items()]

n_asst = sum(1 for _, m in messages if m['role'] == 'assistant')
n_user = sum(1 for _, m in messages if m['role'] == 'user')
print("  assistant turns: " + str(n_asst) + "  |  user turns: " + str(n_user))

# 3. Identify SILENT_BASH turns
silent_bash_idx = [
    i for i, (_, m) in enumerate(messages)
    if m['role'] == 'assistant'
    and not m['has_text']
    and 'Bash' in m['tool_names']
]
print("\nSILENT_BASH turns identified: " + str(len(silent_bash_idx)))

# 4. Classify each SILENT_BASH by its immediately preceding message
AUTONOMOUS    = 'AUTONOMOUS'
USER_PROMPTED = 'USER_PROMPTED'
AMBIGUOUS     = 'AMBIGUOUS'

def classify_preceding(idx):
    if idx == 0:
        return AMBIGUOUS
    _, prev = messages[idx - 1]
    if prev['role'] != 'user':
        return AMBIGUOUS
    if prev['is_tool_result']:
        return AUTONOMOUS
    if prev['is_human']:
        return USER_PROMPTED
    return AMBIGUOUS

classifications = [classify_preceding(i) for i in silent_bash_idx]
counts = Counter(classifications)
total  = len(classifications)

print("\n" + "="*60)
print("  SILENT_BASH ORIGIN SPLIT  (n=" + str(total) + ")")
print("="*60)
for lbl in [AUTONOMOUS, USER_PROMPTED, AMBIGUOUS]:
    n   = counts[lbl]
    pct = round(n / total * 100, 1) if total > 0 else 0
    bar = '#' * int(n / total * 40) if total > 0 else ''
    print("  " + lbl.ljust(16) + str(n).rjust(4) + "  (" + str(pct) + "%)  " + bar)

auto_pct = counts[AUTONOMOUS]    / total * 100 if total > 0 else 0
user_pct = counts[USER_PROMPTED] / total * 100 if total > 0 else 0
ap = str(round(auto_pct))
up = str(round(user_pct))

print("\n  Interpretation:")
if auto_pct >= 70:
    print("  OK: " + ap + "% autonomous.")
    print("  Finding Two holds without qualification.")
elif auto_pct >= 40:
    print("  MIXED: " + ap + "% autonomous / " + up + "% user-prompted.")
    print("  ~" + up + "% were user-directed silence.")
    print("  Silence finding holds; autonomy claim needs nuance.")
else:
    print("  REVISE: " + up + "% user-prompted.")
    print("  Reframe Finding Two as 'responding without narrating'.")

# 5. ASSERTED -> SILENT_BASH chains
print("\n" + "="*60)
print("  ASSERTED -> SILENT_BASH CHAIN ANALYSIS")
print("="*60)

# message_id in full_df is the integer row position (0-based index)
# Map these to positions in our messages list by matching speaking-turn order
asserted_row_nums = full_df[full_df['classification'] == 'ASSERTED']['message_id'].tolist()

speaking_turns = [
    (i, tid) for i, (tid, m) in enumerate(messages)
    if m['role'] == 'assistant' and m.get('has_text')
]
print("Speaking turns in messages list: " + str(len(speaking_turns)))
print("Expected (full_df rows):         " + str(len(full_df)))

asserted_positions = set()
for row_num in asserted_row_nums:
    if row_num < len(speaking_turns):
        msg_pos, tid = speaking_turns[row_num]
        asserted_positions.add(msg_pos)

print("ASSERTED positions mapped: " + str(len(asserted_positions)))

chains = []
for i, (tid, m) in enumerate(messages):
    if i not in asserted_positions:
        continue
    for j in range(i + 1, len(messages)):
        tid_j, m_j = messages[j]
        if m_j['role'] == 'user':
            continue
        is_silent_bash = (
            not m_j.get('has_text')
            and 'Bash' in m_j.get('tool_names', [])
        )
        if is_silent_bash:
            user_type = classify_preceding(j)
            chains.append({
                'asserted_id': tid,
                'user_type':   user_type,
                'bash_id':     tid_j,
                'distance':    j - i,
            })
        break

chain_counts = Counter(c['user_type'] for c in chains)
print("ASSERTED->SILENT_BASH chains found: " + str(len(chains)))
print()
for lbl in [AUTONOMOUS, USER_PROMPTED, AMBIGUOUS]:
    print("  " + lbl.ljust(16) + "  " + str(chain_counts[lbl]))

print("\n  Detail:")
for c in chains:
    a = c['asserted_id'][:20]
    b = c['bash_id'][:20]
    u = c['user_type']
    d = str(c['distance'])
    print("    asserted: " + a + "  user: " + u.ljust(16) + "  bash: " + b + "  dist: " + d)

n_auto = chain_counts.get(AUTONOMOUS, 0)
n_up   = chain_counts.get(USER_PROMPTED, 0)
nc     = len(chains)
print("\n  Implication for Finding One (61.3% execution rate):")
if nc == 0:
    print("  No ASSERTED->SILENT_BASH chains found.")
    print("  The 7 transitions in Cell 32 had an intervening speaking turn.")
elif n_auto == nc:
    print("  All " + str(nc) + " chains were autonomous (tool result in between).")
    print("  Finding One is fully supported without qualification.")
elif n_up > 0:
    print("  " + str(n_up) + " of " + str(nc) + " chains had a human message in between.")
    print("  Finding One holds but note the user endorsed the assertion in those cases.")

In [ ]:
print("Sample message_ids from full_df:")
print(full_df['message_id'].head(5).tolist())
print()
print("Sample ASSERTED message_ids:")
print(full_df[full_df['classification'] == 'ASSERTED']['message_id'].head(5).tolist())

print("Sample tids from messages list (assistant turns):")
asst_tids = [tid for tid, m in messages if m['role'] == 'assistant']
print(asst_tids[:5])

In [ ]:
# Cell 36: Slope Chart, Drift Velocity Speaking-Only vs Full Sequence
# Requires: full_df (Cell 16/23), messages list (Cell 35)

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

# ── 1. Speaking-only drift distances ──────────────────────────────────────
# For each ASSERTED turn, count speaking turns since last GROUNDED speaking turn
labels = full_df['classification'].tolist()
speaking_distances = []
last_grounded = None

for i, lbl in enumerate(labels):
    if lbl == 'GROUNDED':
        last_grounded = i
    elif lbl == 'ASSERTED':
        if last_grounded is None:
            speaking_distances.append(i)
        else:
            speaking_distances.append(i - last_grounded)

print("Speaking-only distances: " + str(sorted(speaking_distances)))
print("  median=" + str(round(float(np.median(speaking_distances)), 1)) +
      "  mean=" + str(round(float(np.mean(speaking_distances)), 1)) +
      "  max=" + str(max(speaking_distances)))

# ── 2. Full-sequence drift distances ──────────────────────────────────────
# For the same 31 ASSERTED turns, count all assistant turns (speaking + silent)
# since last GROUNDED speaking turn — using the messages list from Cell 35

# Build ordered list of assistant turns only (speaking + silent)
asst_sequence = [
    (i_msg, tid, m) for i_msg, (tid, m) in enumerate(messages)
    if m['role'] == 'assistant'
]
# Map messages-list position → index in assistant-only sequence
msg_pos_to_seq_idx = {i_msg: seq_idx for seq_idx, (i_msg, tid, m) in enumerate(asst_sequence)}
# Map assistant sequence index → speaking-turn label (None for silent turns)
seq_idx_to_label = {}
speaking_turns = [(i_msg, tid) for i_msg, (tid, m) in enumerate(messages)
                  if m['role'] == 'assistant' and m.get('has_text')]
for row_idx, (i_msg, tid) in enumerate(speaking_turns):
    seq_idx = msg_pos_to_seq_idx.get(i_msg)
    if seq_idx is not None:
        seq_idx_to_label[seq_idx] = labels[row_idx]

# For each ASSERTED turn, walk back through assistant sequence
# counting all turns until the last GROUNDED speaking turn
asserted_rows = full_df.index[full_df['classification'] == 'ASSERTED'].tolist()
asserted_msg_positions = [speaking_turns[r][0] for r in asserted_rows
                          if r < len(speaking_turns)]
asserted_seq_indices = [msg_pos_to_seq_idx[p] for p in asserted_msg_positions
                        if p in msg_pos_to_seq_idx]

full_distances = []
for seq_idx in asserted_seq_indices:
    steps = 0
    found = False
    for k in range(seq_idx - 1, -1, -1):
        steps += 1
        if seq_idx_to_label.get(k) == 'GROUNDED':
            found = True
            break
    full_distances.append(steps if found else seq_idx)

print("Full-sequence distances:  " + str(sorted(full_distances)))
print("  median=" + str(round(float(np.median(full_distances)), 1)) +
      "  mean=" + str(round(float(np.mean(full_distances)), 1)) +
      "  max=" + str(max(full_distances)))

assert len(speaking_distances) == len(full_distances) == 31, "Expected 31 pairs"

# ── 3. Draw the slope chart ────────────────────────────────────────────────
# Beeswarm: spread dots at the same y-value into a horizontal cluster
def beeswarm_offsets(values, spread=0.06):
    offsets = np.zeros(len(values))
    from collections import defaultdict
    groups = defaultdict(list)
    for i, v in enumerate(values):
        groups[v].append(i)
    for v, idxs in groups.items():
        n = len(idxs)
        if n == 1:
            offsets[idxs[0]] = 0
        else:
            positions = np.linspace(-(n-1)/2, (n-1)/2, n) * spread
            for k, idx in enumerate(idxs):
                offsets[idx] = positions[k]
    return offsets

x_left  = 0 + beeswarm_offsets(speaking_distances)
x_right = 1 + beeswarm_offsets(full_distances)

fig, ax = plt.subplots(figsize=(7, 8))

# Color lines by shift magnitude (full - speaking)
shifts = [full_distances[i] - speaking_distances[i] for i in range(31)]
max_shift = max(shifts) if max(shifts) > 0 else 1
cmap = plt.cm.YlOrRd

for i in range(31):
    color = cmap(0.2 + 0.8 * shifts[i] / max_shift)
    ax.plot(
        [x_left[i], x_right[i]],
        [speaking_distances[i], full_distances[i]],
        color=color, alpha=0.55, linewidth=1.6, zorder=2
    )

# Dots at each end
ax.scatter(x_left,  speaking_distances, color='#4C9BE8', s=55,
           zorder=3, label='Speaking turns only')
ax.scatter(x_right, full_distances,     color='#E05C5C', s=55,
           zorder=3, label='Full 2,262-turn sequence')

# Median lines
sp_med   = np.median(speaking_distances)
full_med = np.median(full_distances)
ax.hlines(sp_med,   -0.15, 0.15, colors='#4C9BE8', linewidths=2.5,
          linestyles='--', zorder=4)
ax.hlines(full_med,  0.85, 1.15, colors='#E05C5C', linewidths=2.5,
          linestyles='--', zorder=4)
ax.text(-0.18, sp_med,   'median ' + str(int(sp_med)),
        va='center', ha='right', fontsize=9, color='#4C9BE8', fontweight='bold')
ax.text( 1.18, full_med, 'median ' + str(int(full_med)),
        va='center', ha='left',  fontsize=9, color='#E05C5C', fontweight='bold')

# Axes and labels
ax.set_xticks([0, 1])
ax.set_xticklabels(['Speaking turns\nonly', 'Full 2,262-turn\nsequence'], fontsize=11)
ax.set_ylabel('Steps since last GROUNDED', fontsize=11)
ax.set_title('Drift Velocity — Each of the 31 ASSERTED Events\n'
             'Measured With Two Different Rulers', fontsize=12, pad=14)
ax.set_xlim(-0.35, 1.35)
ax.set_ylim(-0.5, max(full_distances) + 1)
ax.yaxis.grid(True, linestyle=':', alpha=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend
blue_dot = mlines.Line2D([], [], color='#4C9BE8', marker='o',
                          linestyle='None', markersize=7, label='Speaking turns only')
red_dot  = mlines.Line2D([], [], color='#E05C5C', marker='o',
                          linestyle='None', markersize=7, label='Full sequence')
ax.legend(handles=[blue_dot, red_dot], loc='upper left', fontsize=9, framealpha=0.8)

# Annotation
ax.annotate('Each line = one\nASSERTED event.\nLines slope up when\nsilent turns pushed\nthe distance further.',
            xy=(0.5, max(full_distances) * 0.6),
            fontsize=8.5, color='#555555',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.4', fc='#f9f9f9', ec='#cccccc'))

plt.tight_layout()
plt.savefig('cell36_slope_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cell36_slope_chart.png")

In [ ]:
# Cell 37 — Full assistant turn origin split (all 2,262 turns, speaking + silent)

import json

TRANSCRIPT_PATH = '/content/drive/MyDrive/session-828a165e-full-transcript-20260715.jsonl'

# Parse: collect records, track which assistant message.ids contain text
records = []
msg_has_text = {}

with open(TRANSCRIPT_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except:
            continue
        if rec.get('type') not in ('assistant', 'user'):
            continue
        records.append(rec)
        if rec['type'] == 'assistant':
            mid = rec['message']['id']
            if mid not in msg_has_text:
                msg_has_text[mid] = False
            content = rec['message'].get('content', '')
            if isinstance(content, str) and content.strip():
                msg_has_text[mid] = True
            elif isinstance(content, list):
                for block in content:
                    if isinstance(block, dict) and block.get('type') == 'text':
                        if block.get('text', '').strip():
                            msg_has_text[mid] = True
                            break

# Build deduplicated turn sequence
turns = []
seen = set()
for rec in records:
    key = rec['message']['id'] if rec['type'] == 'assistant' else rec['uuid']
    uid = rec['type'][0] + ':' + key
    if uid not in seen:
        seen.add(uid)
        turns.append((rec['type'], key, rec))

a_total  = sum(1 for t, _, _ in turns if t == 'assistant')
u_total  = sum(1 for t, _, _ in turns if t == 'user')
speaking = sum(1 for t, k, _ in turns if t == 'assistant' and msg_has_text.get(k, False))
silent   = a_total - speaking

print("Unique turns: " + str(len(turns)))
print("  Assistant: " + str(a_total) + "  (speaking: " + str(speaking) + ", silent: " + str(silent) + ")")
print("  User:      " + str(u_total))
print("")

# Classify each assistant turn by preceding user record type
results = {
    'all':      {'autonomous': 0, 'user_prompted': 0, 'no_pred': 0},
    'speaking': {'autonomous': 0, 'user_prompted': 0, 'no_pred': 0},
    'silent':   {'autonomous': 0, 'user_prompted': 0, 'no_pred': 0},
}

for i, (ttype, key, rec) in enumerate(turns):
    if ttype != 'assistant':
        continue
    bucket = 'speaking' if msg_has_text.get(key, False) else 'silent'

    prev_rec = None
    for j in range(i - 1, -1, -1):
        if turns[j][0] == 'user':
            prev_rec = turns[j][2]
            break

    if prev_rec is None:
        results['all']['no_pred'] += 1
        results[bucket]['no_pred'] += 1
        continue

    content = prev_rec['message']['content']
    is_tool = (
        isinstance(content, list) and
        any(isinstance(b, dict) and b.get('type') == 'tool_result' for b in content)
    )
    cat = 'autonomous' if is_tool else 'user_prompted'
    results['all'][cat] += 1
    results[bucket][cat] += 1

def print_split(label, d):
    total = d['autonomous'] + d['user_prompted'] + d['no_pred']
    if total == 0:
        return
    pct_a = round(d['autonomous']   / total * 100, 1)
    pct_u = round(d['user_prompted']/ total * 100, 1)
    print(label + " (n=" + str(total) + "):")
    print("  AUTONOMOUS:    " + str(d['autonomous'])    + "  (" + str(pct_a) + "%)")
    print("  USER_PROMPTED: " + str(d['user_prompted']) + "  (" + str(pct_u) + "%)")
    if d['no_pred']:
        print("  NO PRED:       " + str(d['no_pred']))
    print("")

print_split("ALL ASSISTANT TURNS", results['all'])
print_split("SPEAKING TURNS",      results['speaking'])
print_split("SILENT TURNS",        results['silent'])

human_msgs   = results['all']['user_prompted']
tool_results = results['all']['autonomous']
parallel_overhead = u_total - (human_msgs + tool_results)

print("Implied human messages in session:  " + str(human_msgs))
print("Implied tool result records:        " + str(tool_results))
print("Parallel tool call overhead:        " + str(parallel_overhead))
print("  (= user records that had no matching assistant response,")
print("   because Claude called multiple tools in one turn)")
print("Check: " + str(human_msgs) + " + " + str(tool_results) + " + " + str(parallel_overhead) + " = " + str(human_msgs + tool_results + parallel_overhead) + "  (should equal " + str(u_total) + " user records)")